# Gradio User Interface

In [ ]:
import gradio as gr

# Closing all open ports
gr.close_all()

In [ ]:
import gradio as gr
import pandas as pd

# Beispiel-DataFrame
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Alter": [25, 30, 35],
    "Beruf": ["Ingenieur", "Lehrer", "Designer"]
})

# Interface-Funktion, gibt das DataFrame zurück
def show_df():
    return df

app = gr.Interface(fn=show_df, inputs=[], outputs=gr.Dataframe())

app.launch()


In [ ]:
import gradio as gr


def greet(name):
    return "Hello " + name + "!"


with gr.Blocks() as demo:
    title = gr.HTML("<div style=font-size:100px;'>Stock Predictor</div>")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            load = gr.Button("Load Stock")
            load = gr.Button("Loading Bar")
            #load.click(fn=greet, inputs=name, outputs=output)
            result = gr.Label(value="Successful", label="Result")
        with gr.Column(scale=4):
            img1 = gr.Image("google-test.png")

    with gr.Row():
        with gr.Column(scale=1):
            latestdata = gr.Label(value="Latest Datas of Stock: 2025-06-11", label="Currency")
            nobrands = gr.Label(value="Number of Brands: 61", label="Countin Brands")
            #output = gr.Textbox(label="Output Box")
            #name = gr.Textbox(label="Name")
            drpdwn = gr.Dropdown(label="Choose your Brand:", choices=["Google", "BMW", "Apple"], allow_custom_value=True)
            slider = gr.Slider(label="Number of Days to predict", maximum=10, step=1, value=4)
            prdct = gr.Button("Predict")
        with gr.Column(scale=1):
            start = gr.Label(value="Start Stock in Dataset: 38,343984", label="Start")
            low = gr.Label(value="Lowest Stock: 21,98733", label="Low")
            high = gr.Label(value="Highest Stock: 358,546202", label="High")
            latest = gr.Label(value="Latest Stock: 298,043746", label="Latest")
    news = gr.HTML("<div style=font-size:40px;'>The Latest news of Google:</div>")
    news1 = gr.Label(value="Googles KI wird Modeberater, aber kein Ersatz für Apps", label="Spiegel")
    news2 = gr.Label(value="Google hat jetzt ein neues App-Icon", label="FAZ")
    news3 = gr.Label(value="Mexiko verklagt Google wegen >>Golf von Amerika<<", label="Bild.de")

    #greet_btn = gr.Button("Predict")
    #greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()


In [22]:
gr.close_all()

# Prototype mit Funktionen

In [38]:
import pandas as pd
import kagglehub
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
#import os.path
from pathlib import Path
import time
import gradio as gr
from SPARQLWrapper import SPARQLWrapper, JSON

import numpy as np
from io import BytesIO
from PIL import Image

In [ ]:
#Als Datenrespresentation für die Stocks nehmen?

def image_classifier(inp):
    return {'cat': 0.3, 'dog': 0.7, "viech": 0.9}

demo = gr.Interface(fn=image_classifier, inputs="image", outputs="label")
demo.launch()

In [ ]:
def download(): #unten in den Code noch implementieren
    global path
    global brands
    try: #try download, if fail give the except-statement back
        path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating") #Download latest version
        print("Path to dataset files:", path)
        erg = "<div style=font-size:20px;><div style=background-color:green;><center>Download successul</center></div></div>"

        #Load Data in pandas
        data_path = path+"\World-Stock-Prices-Dataset.csv"
        stockdata = pd.read_csv(data_path)
        #Create DataFrame
        df = pd.DataFrame(stockdata)

        brands = df["Brand_Name"].unique().tolist() #create a list of all brands in the df
        brand_AUSWAHL = brands[39] #Platzhalter für eine spätere Auswahl vom User
        df_apple = df.loc[df["Brand_Name"] == brand_AUSWAHL, ["Date", "Close", "Brand_Name"]]
    except:
        erg = "<div style=font-size:20px;><div style=background-color:red;><center>Download failed</center></div></div>"
        #df = pd.DataFrame({"Fail": ["Fail"]})
    return erg

In [ ]:
def updatedata():
    path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating")
    return "<div style=font-size:20px;><div style=background-color:green;><center>Update Successul</center></div></div>"



def predict(brand, days):
    df_brand = df.loc[df["Brand_Name"] == brand, ["Date", "Close", "Brand_Name"]]
    
    brand_first = str(df_brand["Close"].iloc[-1])
    brand_latest = str(df_brand["Close"].iloc[0])
    max_stock = df_brand["Close"].max()
    min_stock = df_brand["Close"].min()


    df_brand_preproc = df_brand.rename(columns={"Date": "timestamp", "Close": "target", "Brand_Name": "item_id"})
    df_brand_preproc["item_id"] = df_brand_preproc['item_id'].astype("string")
    df_brand_preproc["timestamp"] = df_brand_preproc['timestamp'].astype("string")
    timecut = df_brand_preproc["timestamp"].str.slice(stop=10) #Cut hh:mm:ss and timezone
    df_brand_preproc["timestamp"] = timecut
    df_brand_preproc["timestamp"] = pd.to_datetime(timecut) #convert string into datetime64
    df_brand_reordered =  df_brand_preproc[['item_id', 'timestamp', 'target']] #Reordering columns
    df_irregular = TimeSeriesDataFrame(
        pd.DataFrame(df_brand_reordered)
    )
    df_regular = df_irregular.convert_frequency(freq="D")
    df_filled = df_regular.fill_missing_values()
    data = TimeSeriesDataFrame.from_data_frame(
        df = df_filled,
        id_column="item_id",
        timestamp_column="timestamp"
    )

    prediction_length = days
    train_data, test_data = data.train_test_split(prediction_length)

    predictor = TimeSeriesPredictor(prediction_length=prediction_length, freq="D").fit(
        train_data, presets="bolt_base", hyperparameters={"Chronos": {"fine_tune": True, "fine_tune_lr": 1e-3, "fine_tune_steps": 2000}},
        time_limit=1,
    )

    predictions = predictor.predict(train_data)
    prdct_img = predictor.plot(
        data=data,
        predictions=predictions,
        item_ids=data.item_ids[:2],
        max_history_length=200,
    );

    buf = BytesIO()
    prdct_img.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    image = Image.open(buf) 

    return brand_first, brand_latest, max_stock, min_stock, np.array(image)

def knowledgegraph(user_choice):
    brands = []
    data = pd.read_csv(data_path)
    arr = data["Brand_Name"].unique() #gebe jede firma ohne dubletten an
    for i in arr:
        brands.append(i.replace(" ", "-")) #lösche leerzeichen und ersetze diese durch bindestriche

    df_brands = pd.DataFrame({"brands" : brands}) #neues data frame erstellen mir den firmen
    wikidata_elements = ["Q56276186", "Q926699", "Q3295867", "Q3895", "Q194360", "Q157064", "Q328840", "Q11463", "Q157062", "Q173395", "Q192314", "Q504998", "Q63327", "Q1141173", "Q8074134", "Q53268", "Q1057464", 
                        "Q38076", "Q864407", "Q489921", "Q333718", "Q780442", "Q212405", "Q16972754", "Q459477", "Q159433", "Q170416", "Q63335", "Q907311", "Q128896", "Q188273", "Q7501150", "Q503308", "Q223127", 
                        "Q3884", "Q312", "Q483915", "Q1046951", "Q95", "Q689141", "Q17460900", "Q7414", "Q67186598", "Q8093", "Q188920", "Q2283", "Q715583", "Q2842931", "Q609466", "Q96095585", "Q26678", 
                        "Q465751", "Q868666", "Q40993", "Q9584", "Q941127", "Q182477", "Q37158", "Q478214", "Q918", "Q174310", "Q30258651"]
    df_brands["wikidata"] = wikidata_elements #füge die wikidata identifier als neue spalte hinzu

    company = df_brands.loc[df_brands["brands"] == user_choice.replace(" ", "-")]
    brand_identifier = company["wikidata"].loc[company.index[0]]

    query_start = "SELECT ?officialname ?logo ?inception ?totalassets ?revenue ?netprofit ?operatingincome ?marketcapitalization\n WHERE {"
    query_order = "wd:"+brand_identifier+" wdt:P1448 ?officialname;\n wdt:P154 ?logo;\n wdt:P571 ?inception;\n wdt:P2403 ?totalassets;\n wdt:P2139 ?revenue;\n wdt:P2295 ?netprofit;\n wdt:P3362 ?operatingincome;\n wdt:P2226 ?marketcapitalization.\n"
    query_end = "}"
    query1 = query_start+query_order+query_end

    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    # From https://www.wikidata.org/wiki/Wikidata:SPARQL_query_service/queries/examples#Cats
    sparql.setQuery(query1)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    officialname = results['results']['bindings'][0]["officialname"]["value"]
    logo = results['results']['bindings'][0]["logo"]["value"]
    logo_html = "<center><img src='"+logo+"' width='100' height='100'></center></img>"
    inception = results['results']['bindings'][0]["inception"]["value"]
    inception_short = inception[:10]
    totalassets = "{:,}".format(int(results['results']['bindings'][0]["totalassets"]["value"]))+" €"
    revenue = "{:,}".format(int(results['results']['bindings'][0]["revenue"]["value"]))+" €"
    netprofit = "{:,}".format(int(results['results']['bindings'][0]["netprofit"]["value"]))+" €"
    operatingincome = "{:,}".format(int(results['results']['bindings'][0]["operatingincome"]["value"]))+" €"
    marketcaptl = "{:,}".format(int(results['results']['bindings'][0]["marketcapitalization"]["value"]))+" €"


    return officialname, logo_html, inception_short, totalassets, revenue, netprofit, operatingincome, marketcaptl

with gr.Blocks() as demo:
    path = kagglehub.dataset_download("nelgiriyewithana/world-stock-prices-daily-updating") #Download aktuellste version
    global data_path #als globale variable deklarieren um in den functionen auch drauf zugreifen zu können
    data_path = path+"\World-Stock-Prices-Dataset.csv" 
    stockdata = pd.read_csv(data_path) #Daten in Pandas laden
    df = pd.DataFrame(stockdata) #DataFrame mit den Daten erzeugen

    date_clean = df["Date"].str.slice(stop=10) #Timezone entfernen
    date_first = date_clean.iloc[-1] #Erstes Datum im Datensatz
    date_latest = date_clean.iloc[0] #Letztes/Aktuellsts Datum im Datensatz

    brands = df["Brand_Name"].unique().tolist() #create a list of all brands in the df

    
    title = gr.HTML("<div style=font-size:100px;'>Stock Predictor</div>")
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            load = gr.Button("Update Stock")
            dataupdate = gr.HTML("<div style=font-size:20px;><div style=background-color:grey;><center>Data not updated yet</center></div></div>")
            load.click(fn=updatedata, inputs=[], outputs=dataupdate)
        with gr.Column(scale=4):
            df_presentator = df[["Date","Close","Brand_Name","Country"]]
            gr.HTML("<div style=font-size:30px;><u>10 Samples out of the Data:</u></div>")
            gr.DataFrame(df_presentator.sample(10))

    with gr.Row():
        with gr.Column(scale=1):
            nobrands = gr.Label(value="Number of Brands in Dataset: "+str(len(brands)), label="Counting Brands")
            #output = gr.Textbox(label="Output Box")
            #name = gr.Textbox(label="Name")
            
        with gr.Column(scale=1):
            latestdata = gr.Label(value="Latest Data of Stock: "+date_latest, label="Currency")

    drpdwn = gr.Dropdown(label="Choose your Brand:", choices=brands, interactive=True)
    slider = gr.Slider(label="Number of Days to predict", minimum=1, maximum=50, step=1, value=3, interactive=True)
    prdct = gr.Button("Predict")
    
    with gr.Row():
        with gr.Column(scale=1):
            brand_logo = gr.HTML() #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_titel = gr.Label(value="Official Brand Titel", label="Official Brand Titel") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_inception = gr.Label(value="Inception", label="Inception") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
        with gr.Column(scale=1):
            brand_mrktcapt = gr.Label(value="Brand Market Capitalization (Börsenwert)", label="Brand Market Capitalization (Börsenwert)") #Official Name & Logo + Gründungsdatum + Market Capitalization (Börsenwert)
    
    img1 = gr.Image()
    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            start = gr.Label(value="Start Stock in Dataset", label="Start Stock")
            low = gr.Label(value="Lowest Stock", label="Historically Lowest Stock")
            
        with gr.Column(scale=1):
            latest = gr.Label(value="Latest Stock", label="Current Stock")
            high = gr.Label(value="Highest Stock", label="Historically Highest Stock")
    prdct.click(fn=predict, inputs=[drpdwn, slider], outputs=[start, latest, high, low, img1])
    
    with gr.Row(equal_height=True): #Informationen zum Unternehmen aus Wikidata
        with gr.Column(scale=1):
            assets = gr.Label(value="Total Assets", label="Total Assets (Vermögen)")
            revenue = gr.Label(value="Total Revenue", label="Total Revenue (Gesamtumsatz)")
            
        with gr.Column(scale=1):
            proft = gr.Label(value="Net Profit", label="Net Profit (Bilanzgewinn)")
            income = gr.Label(value="Operating Income", label="Operating Income (Betriebsgewinn)")

    prdct.click(fn=knowledgegraph, inputs=drpdwn, outputs=[brand_titel, brand_logo, brand_inception, assets, revenue, proft, income, brand_mrktcapt])

    news = gr.HTML("<div style=font-size:40px;'>The Latest news of Google:</div>")
    news1 = gr.Label(value="Googles KI wird Modeberater, aber kein Ersatz für Apps", label="Spiegel")
    news2 = gr.Label(value="Google hat jetzt ein neues App-Icon", label="FAZ")
    news3 = gr.Label(value="Mexiko verklagt Google wegen >>Golf von Amerika<<", label="Bild.de")

    #greet_btn = gr.Button("Predict")
    #greet_btn.click(fn=greet, inputs=name, outputs=output, api_name="greet")

demo.launch()

<>:103: SyntaxWarning: invalid escape sequence '\W'
<>:103: SyntaxWarning: invalid escape sequence '\W'
C:\Users\s3phi\AppData\Local\Temp\ipykernel_7688\2087931858.py:103: SyntaxWarning: invalid escape sequence '\W'
  data_path = path+"\World-Stock-Prices-Dataset.csv"


* Running on local URL:  http://127.0.0.1:7902
* To create a public link, set `share=True` in `launch()`.


Beginning AutoGluon training... Time limit = 1s
AutoGluon will save models to 'c:\workspace\CrimeMap\AutogluonModels\ag-20250627_193340'
=================== System Info ===================
AutoGluon Version:  1.3.0
Python Version:     3.12.1
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          8
GPU Count:          0
Memory Avail:       1.56 GB / 15.70 GB (9.9%)
Disk Space Avail:   12.70 GB / 475.50 GB (2.7%)
Setting presets to: bolt_base

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WQL,
 'freq': 'D',
 'hyperparameters': {'Chronos': {'fine_tune': True,
                                 'fine_tune_lr': 0.001,
                                 'fine_tune_steps': 2000}},
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 10,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': True,
 